## **Loading markdown or txt files**

In [1]:
from pathlib import Path

# 当前文件所在路径（notebook / script 都适用）
BASE_DIR = Path.cwd()

# 自动找到 project root
if (BASE_DIR / "data").exists():
    project_root = BASE_DIR
else:
    project_root = BASE_DIR.parent

# 你的目标目录
md_dir = project_root / "data/processed/pdf_parsed"

print("Project root:", project_root)
print("MD dir:", md_dir)
print("Exists:", md_dir.exists())

Project root: /Users/promab/anaconda_projects/email_agent
MD dir: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed
Exists: True


In [2]:
txt_files = list(md_dir.rglob("*.txt"))
print(f"Found {len(txt_files)} txt files")

Found 2 txt files


In [3]:
md_files = list(md_dir.glob("*.md"))
print(f"Found {len(md_files)} markdown files")

Found 13 markdown files


In [4]:
from langchain_community.document_loaders import TextLoader

all_docs = []

all_files = md_files + txt_files # 合并markdown和txt files 

for file in all_files:
    loader = TextLoader(str(file), encoding="utf-8")
    docs = loader.load()

    for d in docs:
        d.metadata["file_name"] = file.name
        d.metadata["source_path"] = str(file)
        d.metadata["file_type"] = file.suffix # 加上文件suffix用于识别文件

    all_docs.extend(docs)

print(f"Total docs loaded: {len(all_docs)}")

Total docs loaded: 15


In [5]:
doc = all_docs[0]
print(doc.page_content)
print(doc.metadata)

# **Yeast Protein Expression**

## **Yeast Protein Expression Overview**

This service provides protein expression using yeast systems, offering High-level expression, Competitive pricing, Fast turnaround time, Scalable production as necessary and tag removal upon request

## **Yeast Protein Expression Standard Service Workflow (Two Phases)**

### **Phase I**

Phase I of Yeast Protein Expression focuses on constructing and validating the expression system:

* Amplification or isolation of the gene of interest (GOI) from a customer-provided construct and subcloning into a transfer vector  
* Sequence confirmation of the insert  
* Mini-induction to over-express the target protein in Pichia  
* Expression testing using SDS-PAGE and Western blot (Customer must provide antibody to protein of interest, if desired)

**Estimated time:** 4–6 weeks  
**Cost:** $3,000 (due at end of phase)

### **Phase II**

Phase II of Yeast Protein Expression focuses on protein production and purification:

* 

## **Chunking**

In [7]:
# 定义markdown文件的切分函数
# 结合markdown结构化切分和细粒度的recursive切分
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def split_markdown(md_text: str, chunk_size=800, chunk_overlap=100):
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )
    
    md_docs = header_splitter.split_text(md_text)
    
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    final_chunks = []
    
    for doc in md_docs:
        sub_chunks = char_splitter.split_text(doc.page_content)
        for chunk in sub_chunks:
            final_chunks.append({
                "content": chunk,
                "metadata": doc.metadata
            })
    
    return final_chunks

In [8]:
# 定义txt文件的切分函数
# 结合自定义的bloc进行切分
import re

def split_txt(text: str):
    """
    Split based on actual schema in your CAR-T file
    Priority:
    1. PRODUCT
    2. PRODUCT_GROUP
    3. Other blocks
    """

    patterns = [
        r"\[PRODUCT\][\s\S]*?\[END_PRODUCT\]",
        r"\[PRODUCT_GROUP\][\s\S]*?\[END_PRODUCT_GROUP\]",
        r"\[(OVERVIEW|PLATFORM|CAPABILITY|WORKFLOW|CELL_TYPES|SUMMARY)\][\s\S]*?\[END_\1\]",
    ]

    chunks = []

    for pattern in patterns:
        matches = re.findall(pattern, text)
        chunks.extend(matches)

    # 如果一个都没匹配到 → fallback
    if not chunks:
        chunks = [text.strip()]

    return [c.strip() for c in chunks]

In [9]:
# 抓取metadata
def extract_metadata(chunk):
    meta = {}

    # block type
    type_match = re.search(r"\[(\w+)\]", chunk)
    if type_match:
        meta["block_type"] = type_match.group(1)

    # title
    title_match = re.search(r"title:\s*(.*)", chunk)
    if title_match:
        meta["title"] = title_match.group(1).strip()

    # product info
    catalog = re.search(r"catalog_no:\s*(.*)", chunk)
    if catalog:
        meta["catalog_no"] = catalog.group(1).strip()

    antigen = re.search(r"target_antigen:\s*(.*)", chunk)
    if antigen:
        meta["target"] = antigen.group(1).strip()

    name = re.search(r"name:\s*(.*)", chunk)
    if name:
        meta["name"] = name.group(1).strip()

    return meta

In [10]:
chunked_docs = []

for doc in all_docs:
    text = doc.page_content
    base_meta = doc.metadata.copy()

    if not text.strip():
        continue

    if base_meta["file_type"] == ".txt":
        chunks = split_txt(text)

        for i, c in enumerate(chunks):
            meta = base_meta.copy()
            meta.update(extract_metadata(c))
            meta["chunk_id"] = i

            chunked_docs.append({
                "content": c,
                "metadata": meta
            })

    elif base_meta["file_type"] == ".md":
        md_chunks = split_markdown(text)

        for i, item in enumerate(md_chunks):
            meta = base_meta.copy()
            meta.update(item["metadata"])
            meta["chunk_id"] = i

            chunked_docs.append({
                "content": item["content"],
                "metadata": meta
            })

print(f"Total chunks: {len(chunked_docs)}")

Total chunks: 362


In [11]:
# 随机抽样检查chunking的质量
import random

samples = random.sample(chunked_docs, 5)

for i, s in enumerate(samples):
    print("=" * 80)
    print(f"Sample {i}")
    
    print("\n--- METADATA ---")
    for k, v in s["metadata"].items():
        print(f"{k}: {v}")
    
    print("\n--- CONTENT ---")
    print(s["content"]) 

Sample 0

--- METADATA ---
source: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt
file_name: CAR-T Products Brochure.txt
source_path: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt
file_type: .txt
block_type: PRODUCT
catalog_no: PM-CAR1077
target: CLDN18.2
chunk_id: 49

--- CONTENT ---
[PRODUCT]
catalog_no: PM-CAR1077
target_antigen: CLDN18.2
costimulatory_domain: 4-1BB
construct: CLDN18.2scFv-TM-4-1BB-CD3ζ
price_usd: 1259
[END_PRODUCT]
Sample 1

--- METADATA ---
source: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/mRNA-LNP Products & Services Brochure.txt
file_name: mRNA-LNP Products & Services Brochure.txt
source_path: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/mRNA-LNP Products & Services Brochure.txt
file_type: .txt
block_type: PRODUCT
catalog_no: PM-LNP-0106
name: VISTA-TF tag
chunk_id: 88

--- CONTENT ---
[PRODUCT]
name: VISTA-T

In [12]:
md_chunks = [d for d in chunked_docs if d["metadata"]["file_type"] == ".md"]

print(f"Total markdown chunks: {len(md_chunks)}")

samples = random.sample(md_chunks, 5)

for i, s in enumerate(samples):
    print("=" * 80)
    print(f"Sample {i}")
    
    print("\n--- METADATA ---")
    for k, v in s["metadata"].items():
        print(f"{k}: {v}")
    
    print("\n--- CONTENT ---")
    print(s["content"])

Total markdown chunks: 95
Sample 0

--- METADATA ---
source: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/E. coli Protein Expression.md
file_name: E. coli Protein Expression.md
source_path: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/E. coli Protein Expression.md
file_type: .md
Header 2: **E. coli Protein Expression Standard Service Workflow**
Header 3: **Phase II**
chunk_id: 2

--- CONTENT ---
* 1L bacterial culture induced with IPTG and harvested for protein purification
* Protein purification using NiNTA (for 6×His-tagged proteins) or glutathione beads (for GST-tagged proteins) beads either under native or denatured conditions
* Solubility is not guaranteed  
**Estimated time:** 3 weeks
**Cost:** $1,300 (due at initiation)  
Yield ranges from 200 μg to 50 mg/L of culture with average yield of 3 mg/L  
**E. coli Protein Expression Total Time and Cost**  
* **Overall duration:** 6–8 weeks
* **Total cost:** $3,300
Sample 1

--- METADAT

In [13]:
from langchain_core.documents import Document

docs_for_vectorstore = []

for item in chunked_docs:
    docs_for_vectorstore.append(
        Document(
            page_content=item["content"],
            metadata=item["metadata"]
        )
    )

print(f"Documents ready for vector store: {len(docs_for_vectorstore)}")
print(docs_for_vectorstore[0])

Documents ready for vector store: 362
page_content='This service provides protein expression using yeast systems, offering High-level expression, Competitive pricing, Fast turnaround time, Scalable production as necessary and tag removal upon request' metadata={'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/Yeast Protein Expression.md', 'file_name': 'Yeast Protein Expression.md', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/Yeast Protein Expression.md', 'file_type': '.md', 'Header 1': '**Yeast Protein Expression**', 'Header 2': '**Yeast Protein Expression Overview**', 'chunk_id': 0}


## **Vectorization**

In [14]:
# 读取openai key
from dotenv import load_dotenv
import os

# 当前路径
BASE_DIR = Path.cwd()

# 自动找到 project root
if (BASE_DIR / ".env").exists():
    env_path = BASE_DIR / ".env"
else:
    env_path = BASE_DIR.parent / ".env"

# 加载
load_dotenv(env_path)

# 读取
api_key = os.getenv("OPENAI_API_KEY")

print("API KEY:", api_key[:10], "...")

API KEY: sk-proj-QH ...


In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 1) embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2) persist dir
persist_dir = project_root / "data" / "vectorstore" / "chroma_docs"
persist_dir.mkdir(parents=True, exist_ok=True)

# 3) create vector store
vectorstore = Chroma.from_documents(
    documents=docs_for_vectorstore,
    embedding=embeddings,
    persist_directory=str(persist_dir),
)

print("Vector store created!")
print("Saved to:", persist_dir)
print("Number of chunks:", vectorstore._collection.count())

Vector store created!
Saved to: /Users/promab/anaconda_projects/email_agent/data/vectorstore/chroma_docs
Number of chunks: 362


#### 做similarity_research测试向量数据库

In [16]:
results = vectorstore.similarity_search("What CAR-T products target CD19?", k=5)

for i, doc in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1000])

Result 1
Metadata: {'file_type': '.txt', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'target': 'CD19 + CD22', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'block_type': 'PRODUCT', 'catalog_no': 'PM-CAR1049', 'file_name': 'CAR-T Products Brochure.txt', 'chunk_id': 30}
[PRODUCT]
catalog_no: PM-CAR1049
target_antigen: CD19 + CD22
costimulatory_domain: 4-1BB
construct: CD19scFv-CD22scFv-TM-4-1BB-CD3ζ-T2A-tEGFR
price_usd: 1259
[END_PRODUCT]
Result 2
Metadata: {'file_type': '.txt', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'file_name': 'CAR-T Products Brochure.txt', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'target': 'CD19', 'chunk_id': 24, 'block_type': 'PRODUCT', 'catalog_no': 'PM-CAR1067'}
[PRODUCT]
catalog_no: P

In [17]:
vectorstore.similarity_search("CAR-T products with CD28 costimulatory domain", k=5)

[Document(id='2abde8fa-9f19-4ec3-929c-da13960f06ac', metadata={'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'block_type': 'PRODUCT', 'file_name': 'CAR-T Products Brochure.txt', 'chunk_id': 32, 'target': 'CD20', 'file_type': '.txt', 'catalog_no': 'PM-CAR1062'}, page_content='[PRODUCT]\ncatalog_no: PM-CAR1062\ntarget_antigen: CD20\ncostimulatory_domain: CD28\nconstruct: CD20scFv-TM28-CD28-CD3ζ\nprice_usd: 1259\n[END_PRODUCT]'),
 Document(id='e9a7759b-9038-42ff-9cd7-ff12f9704278', metadata={'file_name': 'CAR-T Products Brochure.txt', 'target': 'N/A', 'block_type': 'PRODUCT', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'file_type': '.txt', 'chunk_id': 87, 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T

In [18]:
vectorstore.similarity_search("CAR-T products targeting CD22", k=5)

[Document(id='51fb9216-3bdf-4192-957e-b7130eb14ccc', metadata={'chunk_id': 34, 'file_type': '.txt', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'block_type': 'PRODUCT', 'file_name': 'CAR-T Products Brochure.txt', 'target': 'CD22', 'catalog_no': 'PM-CAR1029', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt'}, page_content='[PRODUCT]\ncatalog_no: PM-CAR1029\ntarget_antigen: CD22\ncostimulatory_domain: CD28\nconstruct: CD22scFv-TM28-CD28-CD3ζ\nprice_usd: 1259\n[END_PRODUCT]'),
 Document(id='30672ec1-3a91-474b-b694-51f20a93e643', metadata={'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'target': 'CD22', 'block_type': 'PRODUCT', 'file_name': 'CAR-T Products Brochure.txt', 'chunk_id': 37, 'catalog_no': 'PM-CAR1081', 'file_type': '.txt', 'source': '/Users/promab/anaconda_projects/email_agent/da

In [19]:
vectorstore.similarity_search("Which CAR-T products cost 1259 USD?", k=5)

[Document(id='e1174816-5237-415f-bb5d-905d9d50faa6', metadata={'catalog_no': 'PM-CAR1049', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'file_name': 'CAR-T Products Brochure.txt', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'chunk_id': 30, 'block_type': 'PRODUCT', 'file_type': '.txt', 'target': 'CD19 + CD22'}, page_content='[PRODUCT]\ncatalog_no: PM-CAR1049\ntarget_antigen: CD19 + CD22\ncostimulatory_domain: 4-1BB\nconstruct: CD19scFv-CD22scFv-TM-4-1BB-CD3ζ-T2A-tEGFR\nprice_usd: 1259\n[END_PRODUCT]'),
 Document(id='db3b6af8-641a-4568-8f5b-0507892cfada', metadata={'chunk_id': 31, 'file_type': '.txt', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'catalog_no': 'PM-CAR1050', 'target': 'CD19 + CD22', 'block_type': 'PRODUCT', 'source': '/Users/promab/anaconda_projects/email_agent/data/p

## **Building RAG MVP**

| 模块                    | 作用                        |
| --------------------- | ------------------------- |
| `langchain_core`      | 核心抽象（Prompt / Runnable） |
| `langchain_community` | loaders / vectorstores    |
| `langchain`           | 高层封装（越来越薄）                |


In [20]:
# converting vectorstote to retriever
# 把「向量数据库」包装成一个可以用自然语言查询的接口
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

| 类型                         | 作用                |
| -------------------------- | ----------------- |
| similarity                 | 纯相似度（默认）          |
| mmr                        | 多样性（避免重复 chunk）🔥 |
| similarity_score_threshold | 只返回高于阈值的          |


In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",   # 成本低，速度快，很适合你这个agent
    temperature=0          # 保持稳定输出（很重要！）
)

In [31]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a professional biotech assistant.

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "I don't know."

When listing products, use this format:
- Catalog No
- Target Antigen
- Costimulatory Domain
- Construct
- Price

Context:
{context}

Question:
{question}
""")

# RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# test
rag_chain.invoke("Do you have CD19 CAR-T?")

'Yes, we have CD19 CAR-T products. Here are the details:\n\n1. \n- Catalog No: PM-CAR1067\n- Target Antigen: CD19\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-TM-FLAG-4-1BB-CD3ζ /// tEGFR\n- Price: $1259\n\n2. \n- Catalog No: PM-CAR1006\n- Target Antigen: CD19\n- Costimulatory Domain: CD28\n- Construct: iCas9-T2A-CD19scFv-TM-CD28-CD3ζ\n- Price: $1259\n\n3. \n- Catalog No: PM-CAR1087\n- Target Antigen: CD19\n- Costimulatory Domain: CD28 + 4-1BB\n- Construct: CD19scFv-TF-CD28-4-1BB-CD3ζ\n- Price: $1259\n\n4. \n- Catalog No: PM-CAR1066\n- Target Antigen: CD19\n- Costimulatory Domain: CD28\n- Construct: CD19scFv-TM-FLAG-CD28-CD3ζ /// tEGFR\n- Price: $1259'

In [29]:
#  检查模型到底使用了哪些docs
docs = retriever.invoke("Do you have CD19 CAR-T?")

for i, d in enumerate(docs, 1):
    print(f"===== Doc {i} =====")
    print(d.page_content)
    print(d.metadata)
    print()

===== Doc 1 =====
[PRODUCT]
catalog_no: PM-CAR1067
target_antigen: CD19
costimulatory_domain: 4-1BB
construct: CD19scFv-TM-FLAG-4-1BB-CD3ζ /// tEGFR
price_usd: 1259
[END_PRODUCT]
{'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'file_name': 'CAR-T Products Brochure.txt', 'block_type': 'PRODUCT', 'catalog_no': 'PM-CAR1067', 'chunk_id': 24, 'file_type': '.txt', 'target': 'CD19'}

===== Doc 2 =====
[PRODUCT]
catalog_no: PM-CAR1049
target_antigen: CD19 + CD22
costimulatory_domain: 4-1BB
construct: CD19scFv-CD22scFv-TM-4-1BB-CD3ζ-T2A-tEGFR
price_usd: 1259
[END_PRODUCT]
{'target': 'CD19 + CD22', 'file_type': '.txt', 'file_name': 'CAR-T Products Brochure.txt', 'source': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/CAR-T Products Brochure.txt', 'chunk_id': 30, 'source_path': '/Users/promab/an

In [34]:
rag_chain.invoke("Do you have CD19 CAR-T products?")

'Yes, here are the CD19 CAR-T products:\n\n1. \n- Catalog No: PM-CAR1067\n- Target Antigen: CD19\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-TM-FLAG-4-1BB-CD3ζ /// tEGFR\n- Price: 1259\n\n2. \n- Catalog No: PM-CAR1087\n- Target Antigen: CD19\n- Costimulatory Domain: CD28 + 4-1BB\n- Construct: CD19scFv-TF-CD28-4-1BB-CD3ζ\n- Price: 1259\n\n3. \n- Catalog No: PM-CAR1090\n- Target Antigen: CD19\n- Costimulatory Domain: CD28 + 4-1BB\n- Construct: CD19scFv-FLAG-4-1BB-CD3ζ /// GFP\n- Price: 1259'

In [35]:
rag_chain.invoke("What is the price of CD19 CAR-T products?")

'The price of CD19 CAR-T products is $1259.'

In [36]:
rag_chain.invoke("Do you offer CAR-T targeting both CD19 and CD22?")

'Yes, we offer CAR-T products targeting both CD19 and CD22. Here are the details:\n\n1. \n- Catalog No: PM-CAR1049\n- Target Antigen: CD19 + CD22\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-CD22scFv-TM-4-1BB-CD3ζ-T2A-tEGFR\n- Price: $1259\n\n2. \n- Catalog No: PM-CAR1050\n- Target Antigen: CD19 + CD22\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-CD22scFv-TM-4-1BB-CD3ζ-T2A-RQR8\n- Price: $1259\n\n3. \n- Catalog No: PM-CAR1035\n- Target Antigen: CD19 + CD22\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-CD22scFv-TM-4-1BB-CD3ζ\n- Price: $1259'

In [37]:
rag_chain.invoke("Which CD19 CAR-T product uses 4-1BB?")

'- Catalog No: PM-CAR1067\n- Target Antigen: CD19\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-TM-FLAG-4-1BB-CD3ζ /// tEGFR\n- Price: 1259\n\n- Catalog No: PM-CAR1002\n- Target Antigen: CD19\n- Costimulatory Domain: 4-1BB\n- Construct: CD19scFv-TM-4-1BB-CD3ζ\n- Price: 1259\n\n- Catalog No: PM-CAR1087\n- Target Antigen: CD19\n- Costimulatory Domain: CD28 + 4-1BB\n- Construct: CD19scFv-TF-CD28-4-1BB-CD3ζ\n- Price: 1259'